# E03 — Prompt Selection — Analysis

**Research question**: Which prompt instruction set gives the best classification behavior
for `qwen2.5:7b-instruct` when the retrieved evidence context is held fixed?

Resumed after E06 froze `retrieval_v1` (BM25 -> clause_256 -> top-20 candidates ->
`ms-marco-MiniLM-L-12-v2` rerank -> top-5). The original pre-E06 attempt used full-context NDA
text and was stopped at 88/150 cases, superseded before any metric was computed — see
`results/SUPERSEDED_run_E03_prompt_selection_p00.md`. This run uses the identical frozen
`retrieval_v1` top-5 context for every case, across all three prompt variants. Local-only
(`qwen2.5:7b-instruct` via Ollama), $0 hosted spend.

In [1]:
import csv
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "E03_prompt_selection" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
E03 = REPO_ROOT / "experiments/E03_prompt_selection"
RESULTS = E03 / "results"

manifest = json.load(open(E03 / "TRAIN_PROMPT_v1.json"))
retrieved = json.load(open(E03 / "TRAIN_PROMPT_v1_RETRIEVED_retrieval_v1.json"))
summary = json.load(open(RESULTS / "prompt_failure_analysis_summary.json"))
metrics = summary["metrics"]
print("manifest cases:", len(manifest["cases"]))
print("retrieval config:", retrieved["retrieval_config"])

manifest cases: 150
retrieval config: {'method': 'bm25', 'chunk_method': 'clause', 'chunk_size': 256, 'chunk_overlap': 50, 'embedding_model': None, 'candidate_pool_size': 20, 'top_k': 5, 'reranking': True, 'reranker_model': 'cross-encoder/ms-marco-MiniLM-L-12-v2'}


## 1. Verified frozen manifest and retrieval context

In [2]:
from collections import Counter
balance = Counter(c["gold_label"] for c in manifest["cases"])
print("class balance:", dict(balance))
assert balance == Counter({"Entailment": 50, "Contradiction": 50, "NotMentioned": 50})
assert retrieved["retrieval_config"]["method"] == "bm25"
assert retrieved["retrieval_config"]["top_k"] == 5
print("OK -- 150 cases, 50/50/50 balance, retrieval_v1 (BM25 top-5) confirmed")

class balance: {'Contradiction': 50, 'Entailment': 50, 'NotMentioned': 50}
OK -- 150 cases, 50/50/50 balance, retrieval_v1 (BM25 top-5) confirmed


## 2. Prompt definitions and exact differences

P0/P1/P2 share the identical task framing, output schema, and user-message template — only
the `system_prompt` content differs, incrementally:
- **P0** (minimal): states the 3 labels, asks for the compact JSON output. No definitions.
- **P1** (P0 + label definitions): adds one explicit sentence defining each label
  (Entailment/Contradiction/NotMentioned).
- **P2** (P1 + decision procedure): adds a concise 5-step decision procedure on top of P1's
  definitions (no chain-of-thought/explanation requested — same compact output schema).

A pre-existing wording bug was fixed identically across all three before this run: the prompts
said "the full text of the NDA" / "NDA text:", a stale artifact from before E06 resequencing
when E03 used full-context input. Corrected to "excerpts retrieved from the NDA" / "Retrieved
NDA excerpts:" since `context_text` is now the frozen `retrieval_v1` top-5 context, not the
full document — applied uniformly, not part of the P0/P1/P2 independent variable.

In [3]:
import yaml
for v in ("p00", "p01", "p02"):
    cfg = yaml.safe_load(open(REPO_ROOT / f"configs/prompts/classification/classification_{v}.yaml"))
    print(f"--- {v} ({cfg['system_prompt_tokens_cl100k_approx']} tokens approx) ---")
    print(cfg["system_prompt"])
    print()

--- p00 (84 tokens approx) ---
You are given an NDA requirement and relevant excerpts retrieved from the NDA.

Classify the requirement against the excerpts as exactly one of:
- "Entailment"
- "Contradiction"
- "NotMentioned"

Respond with ONLY a JSON object with this exact field:
{
  "label": "Entailment" | "Contradiction" | "NotMentioned"
}


--- p01 (124 tokens approx) ---
You are given an NDA requirement and relevant excerpts retrieved from the NDA.

Classify the requirement against the excerpts as exactly one of:
- "Entailment": the excerpts state or clearly imply that the requirement is met.
- "Contradiction": the excerpts state or clearly imply terms incompatible with the requirement.
- "NotMentioned": the excerpts do not address this requirement at all.

Respond with ONLY a JSON object with this exact field:
{
  "label": "Entailment" | "Contradiction" | "NotMentioned"
}


--- p02 (197 tokens approx) ---
You are given an NDA requirement and relevant excerpts retrieved from the N

## 3. Experimental controls

Fixed across P0/P1/P2: model (`qwen2.5:7b-instruct`, local Ollama), temperature (0.0), the
frozen `retrieval_v1` top-5 context per case, case ordering, output schema
(`{"label": ...}`), and the parser (`evaluation.oracle.parse_oracle_output`, reused unchanged
from E01). Only the prompt wording changed. All 450 predictions (150 cases x 3 prompts) carry
full traceability (run_id, prompt_config_hash, retrieval_config_version, timestamp) —
see `results/run_E03_prompt_selection_p0{0,1,2}.jsonl`.

## 4-8. Side-by-side metrics (P0 / P1 / P2)

In [4]:
import pandas as pd
rows = []
for v in ("p00", "p01", "p02"):
    m = metrics[v]
    rows.append({
        "prompt": v, "accuracy": round(m["accuracy"], 4), "macro_f1": round(m["macro_f1"], 4),
        "entailment_recall": round(m["per_class_recall"]["Entailment"], 4),
        "contradiction_recall": round(m["contradiction_recall"], 4),
        "contradiction_recall_ci95": [round(x, 3) for x in m["contradiction_recall_ci95"]],
        "notmentioned_recall": round(m["per_class_recall"]["NotMentioned"], 4),
        "parse_valid_rate": round(m["parse_valid_rate"], 4),
        "total_retries": m["total_retries"],
        "latency_mean_ms": round(m["latency_ms"]["mean"], 1) if m["latency_ms"]["mean"] else None,
        "latency_median_ms": round(m["latency_ms"]["median"], 1) if m["latency_ms"]["median"] else None,
        "latency_p90_ms": round(m["latency_ms"]["p90"], 1) if m["latency_ms"]["p90"] else None,
        "mean_input_tokens": round(m["mean_input_tokens"], 1) if m["mean_input_tokens"] else None,
        "mean_output_tokens": round(m["mean_output_tokens"], 1) if m["mean_output_tokens"] else None,
    })
df = pd.DataFrame(rows).set_index("prompt")
df

,accuracy,macro_f1,entailment_recall,contradiction_recall,contradiction_recall_ci95,notmentioned_recall,parse_valid_rate,total_retries,latency_mean_ms,latency_median_ms,latency_p90_ms,mean_input_tokens,mean_output_tokens
prompt,,,,,,,,,,,,,
p00,0.5267,0.5070,0.56,0.22,"[0.128, 0.352]",0.80,1.0,0,4855.6,4970.8,5597.9,1141.9,12.6
p01,0.5133,0.4481,0.60,0.06,"[0.021, 0.162]",0.88,1.0,0,4799.3,4854.9,5408.7,1175.9,12.7
p02,0.4800,0.4035,0.52,0.02,"[0.004, 0.105]",0.90,1.0,0,4881.2,4955.5,5605.7,1244.9,12.8


## 9-10. Confusion matrices

In [5]:
for v in ("p00", "p01", "p02"):
    cm = metrics[v]["confusion_matrix"]
    print(f"--- {v} --- rows=gold, cols=predicted, labels={cm['labels']}")
    for label, row in zip(cm["labels"], cm["matrix"]):
        print(f"  {label:>13}: {row}")
    print()

--- p00 --- rows=gold, cols=predicted, labels=['Entailment', 'Contradiction', 'NotMentioned']
     Entailment: [28, 5, 17]
  Contradiction: [2, 11, 37]
   NotMentioned: [6, 4, 40]

--- p01 --- rows=gold, cols=predicted, labels=['Entailment', 'Contradiction', 'NotMentioned']
     Entailment: [30, 0, 20]
  Contradiction: [3, 3, 44]
   NotMentioned: [6, 0, 44]

--- p02 --- rows=gold, cols=predicted, labels=['Entailment', 'Contradiction', 'NotMentioned']
     Entailment: [26, 0, 24]
  Contradiction: [3, 1, 46]
   NotMentioned: [4, 1, 45]



## 11. Contradiction-focused comparison

Contradiction Recall is reported on its own (not folded into a combined risk-sensitive
average), per the reconstruction brief and the project's own prior instructor-feedback fix —
Contradiction is the smallest class (50/150) and the one E01's Oracle already flagged as the
clearest reasoning bottleneck even with perfect evidence.

In [6]:
c_rows = []
for v in ("p00", "p01", "p02"):
    m = metrics[v]
    lo, hi = m["contradiction_recall_ci95"]
    c_rows.append({"prompt": v, "contradiction_recall": round(m["contradiction_recall"], 4),
                    "n": m["contradiction_n"], "ci95_low": round(lo, 4), "ci95_high": round(hi, 4)})
pd.DataFrame(c_rows).set_index("prompt")

,contradiction_recall,n,ci95_low,ci95_high
prompt,,,,
p00,0.22,50,0.1275,0.3524
p01,0.06,50,0.0206,0.1622
p02,0.02,50,0.0035,0.1050


## 12. Prompt-sensitive cases and agreement patterns

In [7]:
rows_csv = list(csv.DictReader(open(RESULTS / "prompt_failure_analysis.csv")))
for r in rows_csv:
    for k in ("p0_correct", "p1_correct", "p2_correct", "prompt_sensitive"):
        r[k] = r[k] == "True"

print("prompt_sensitive_count:", summary["prompt_sensitive_count"], "/", summary["n_cases"])
print("all_correct_count:", summary["all_correct_count"])
print("all_wrong_count:", summary["all_wrong_count"])

def agreement_bucket(r):
    return (r["p0_correct"], r["p1_correct"], r["p2_correct"])

from collections import Counter as C
print("\ncorrectness-pattern counts (p0,p1,p2):")
for pattern, n in sorted(C(agreement_bucket(r) for r in rows_csv).items()):
    print(" ", pattern, "->", n)

prompt_sensitive_count: 34 / 150
all_correct_count: 62
all_wrong_count: 60

correctness-pattern counts (p0,p1,p2):
  (False, False, False) -> 60
  (False, False, True) -> 1
  (False, True, False) -> 3
  (False, True, True) -> 7
  (True, False, False) -> 10
  (True, False, True) -> 2
  (True, True, False) -> 5
  (True, True, True) -> 62


## 13. Retrieval-limited vs. reasoning/prompt-limited failures

Per case: gold evidence spans are checked against the frozen `retrieval_v1` top-5 context's
character offsets (`evaluation.scorer`-style overlap check). `retrieval_contains_gold=None`
marks NotMentioned cases, which have no annotated evidence by construction (E00 semantics) —
any error there is necessarily reasoning/prompt-limited, not a retrieval failure.

In [8]:
print("retrieval_limited errors:", summary["retrieval_limited_count"])
print("reasoning/prompt_limited errors:", summary["reasoning_prompt_limited_count"])
print("ambiguous/mixed:", summary["ambiguous_count"])
print()
print("Contradiction-specific failure families (observed, heuristic-flagged candidates):")
for fam, n in summary["contradiction_failure_families"].items():
    print(f"  {n:>3}  {fam}")

retrieval_limited errors: 6
reasoning/prompt_limited errors: 82
ambiguous/mixed: 0

Contradiction-specific failure families (observed, heuristic-flagged candidates):
   41  Contradiction->NotMentioned (candidate: exception/carve-out clause present)
    3  Contradiction->NotMentioned (NotMentioned/Contradiction confusion)
    3  gold evidence absent from retrieval_v1 top-5 context
    2  Contradiction->Entailment (candidate: exception/carve-out clause present)


**Methodology caveat**: `failure_family` tags (e.g. "candidate: exception/carve-out clause
present") are automated keyword-heuristic flags on the retrieved context text (presence of
"except"/"unless"/"provided that"/etc.), not a confirmed manual reading of every case. They
identify *candidates* for the representative-failure review below, consistent with the
instruction to use only observed categories and not force weak-evidence cases into a
predefined family.

## 14. Representative failures

In [9]:
sensitive_rows = [r for r in rows_csv if r["prompt_sensitive"] and r["gold_label"] == "Contradiction"]
print(f"{len(sensitive_rows)} prompt-sensitive Contradiction cases; showing up to 5:")
for r in sensitive_rows[:5]:
    print(f"\n{r['case_id']}  gold={r['gold_label']}  retrieval_contains_gold={r['retrieval_contains_gold']}")
    print(f"  p0={r['p0_prediction']} p1={r['p1_prediction']} p2={r['p2_prediction']}")
    print(f"  failure_source={r['failure_source']}  failure_family={r['failure_family']}")

11 prompt-sensitive Contradiction cases; showing up to 5:

train::86::nda-20  gold=Contradiction  retrieval_contains_gold=True
  p0=Contradiction p1=NotMentioned p2=NotMentioned
  failure_source=reasoning_prompt_limited  failure_family=Contradiction->NotMentioned (candidate: exception/carve-out clause present)

train::93::nda-7  gold=Contradiction  retrieval_contains_gold=True
  p0=Contradiction p1=NotMentioned p2=NotMentioned
  failure_source=reasoning_prompt_limited  failure_family=Contradiction->NotMentioned (candidate: exception/carve-out clause present)

train::142::nda-7  gold=Contradiction  retrieval_contains_gold=True
  p0=Contradiction p1=NotMentioned p2=NotMentioned
  failure_source=reasoning_prompt_limited  failure_family=Contradiction->NotMentioned (candidate: exception/carve-out clause present)

train::200::nda-7  gold=Contradiction  retrieval_contains_gold=True
  p0=Contradiction p1=NotMentioned p2=NotMentioned
  failure_source=reasoning_prompt_limited  failure_family=Con

## 15. Token/latency overhead — did prompt complexity earn itself?

P2 (decision procedure) carries the most system-prompt tokens; P0 the fewest. Compare the
metric table in section 4-8 against this overhead directly: added instruction length is only
"earned" if it moves Contradiction Recall / Macro-F1 without a major regression elsewhere.

In [10]:
overhead_rows = []
for v in ("p00", "p01", "p02"):
    cfg = yaml.safe_load(open(REPO_ROOT / f"configs/prompts/classification/classification_{v}.yaml"))
    m = metrics[v]
    overhead_rows.append({
        "prompt": v, "system_prompt_tokens_approx": cfg["system_prompt_tokens_cl100k_approx"],
        "mean_input_tokens_actual": round(m["mean_input_tokens"], 1),
        "mean_output_tokens_actual": round(m["mean_output_tokens"], 1),
        "latency_mean_ms": round(m["latency_ms"]["mean"], 1),
        "contradiction_recall": round(m["contradiction_recall"], 4),
        "macro_f1": round(m["macro_f1"], 4),
    })
pd.DataFrame(overhead_rows).set_index("prompt")

,system_prompt_tokens_approx,mean_input_tokens_actual,mean_output_tokens_actual,latency_mean_ms,contradiction_recall,macro_f1
prompt,,,,,,
p00,84,1141.9,12.6,4855.6,0.22,0.5070
p01,124,1175.9,12.7,4799.3,0.06,0.4481
p02,197,1244.9,12.8,4881.2,0.02,0.4035


## 16-17. Final prompt selection and rationale

Selection priority (predeclared, not tuned after seeing results): (1) Contradiction Recall,
(2) Macro-F1, (3) no major Entailment Recall regression, (4) no major NotMentioned Recall
regression, (5) parse validity, (6) latency/token overhead, (7) prompt simplicity — ties broken
toward the simpler/shorter prompt.

In [11]:
# Selection walk-through -- printed for the record, not re-deciding anything already
# reasoned about in the written report.
for v in ("p00", "p01", "p02"):
    m = metrics[v]
    print(f"{v}: contradiction_recall={m['contradiction_recall']:.4f}  macro_f1={m['macro_f1']:.4f}  "
          f"entailment_recall={m['per_class_recall']['Entailment']:.4f}  "
          f"notmentioned_recall={m['per_class_recall']['NotMentioned']:.4f}  "
          f"parse_valid={m['parse_valid_rate']:.4f}")

p00: contradiction_recall=0.2200  macro_f1=0.5070  entailment_recall=0.5600  notmentioned_recall=0.8000  parse_valid=1.0000
p01: contradiction_recall=0.0600  macro_f1=0.4481  entailment_recall=0.6000  notmentioned_recall=0.8800  parse_valid=1.0000
p02: contradiction_recall=0.0200  macro_f1=0.4035  entailment_recall=0.5200  notmentioned_recall=0.9000  parse_valid=1.0000


**Selected prompt and rationale**: see the final report delivered alongside this notebook
and `configs/prompts/classification/classification_prompt_v1.yaml` (the frozen downstream
artifact) for the full selection writeup, applying the priority order above to the numbers
printed here.

## 18. Frozen `classification_prompt_v1` for downstream RAG

Downstream experiments (E05 full-context, E07 standard RAG, E08 RAG failure analysis, E09-E11
agentic work) will default to `classification_prompt_v1`. Prompt selection and architecture
comparison remain separate questions — this freeze does not by itself establish that any given
architecture (full-context / RAG / RAG+agent) is superior; that is E12's job.